<a href="https://colab.research.google.com/github/prajval25soe1004001189-max/praju/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Library Management System
==========================
A simple console-based Library Management System built with Python.

Features:
    1. Add a new book
    2. Search for a book (by title, author, or ISBN)
    3. Issue a book to a member
    4. Return a book
    5. Display all available books
    6. Save records (auto-saved to a JSON file after every change)

Data is persisted to 'library_data.json' in the same folder as this
script, so your records survive between runs.
"""

import json
import os
from datetime import datetime, timedelta

DATA_FILE = os.path.join(os.getcwd(), "library_data.json")
LOAN_PERIOD_DAYS = 14


class Book:
    def __init__(self, isbn, title, author, copies=1, copies_available=None):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.copies = copies
        self.copies_available = copies_available if copies_available is not None else copies

    def to_dict(self):
        return {
            "isbn": self.isbn,
            "title": self.title,
            "author": self.author,
            "copies": self.copies,
            "copies_available": self.copies_available,
        }

    @staticmethod
    def from_dict(d):
        return Book(d["isbn"], d["title"], d["author"], d["copies"], d["copies_available"])

    def __str__(self):
        return (f"ISBN: {self.isbn} | '{self.title}' by {self.author} "
                f"| Available: {self.copies_available}/{self.copies}")


class Library:
    def __init__(self, data_file=DATA_FILE):
        self.data_file = data_file
        self.books = {}      # isbn -> Book
        self.issued = {}     # isbn -> list of loan dicts {member, issue_date, due_date}
        self.load_records()

    # ---------- Persistence ----------
    def load_records(self):
        if os.path.exists(self.data_file):
            try:
                with open(self.data_file, "r") as f:
                    data = json.load(f)
                self.books = {isbn: Book.from_dict(b) for isbn, b in data.get("books", {}).items()}
                self.issued = data.get("issued", {})
            except (json.JSONDecodeError, KeyError):
                print("Warning: data file was corrupted. Starting with an empty library.")
                self.books = {}
                self.issued = {}
        else:
            self.books = {}
            self.issued = {}

    def save_records(self):
        data = {
            "books": {isbn: book.to_dict() for isbn, book in self.books.items()},
            "issued": self.issued,
        }
        with open(self.data_file, "w") as f:
            json.dump(data, f, indent=4)
        print(f"Records saved to '{self.data_file}'.")

    # ---------- Core Features ----------
    def add_book(self, isbn, title, author, copies=1):
        if isbn in self.books:
            self.books[isbn].copies += copies
            self.books[isbn].copies_available += copies
            print(f"Added {copies} more copy(ies) of '{title}'. "
                  f"Total copies: {self.books[isbn].copies}")
        else:
            self.books[isbn] = Book(isbn, title, author, copies)
            print(f"Book '{title}' added successfully.")
        self.save_records()

    def search_books(self, keyword):
        keyword = keyword.lower().strip()
        results = [
            b for b in self.books.values()
            if keyword in b.title.lower()
            or keyword in b.author.lower()
            or keyword in b.isbn.lower()
        ]
        return results

    def issue_book(self, isbn, member_name):
        book = self.books.get(isbn)
        if not book:
            print("No book found with that ISBN.")
            return
        if book.copies_available <= 0:
            print(f"Sorry, '{book.title}' has no available copies right now.")
            return

        book.copies_available -= 1
        issue_date = datetime.now()
        due_date = issue_date + timedelta(days=LOAN_PERIOD_DAYS)
        loan = {
            "member": member_name,
            "issue_date": issue_date.strftime("%Y-%m-%d"),
            "due_date": due_date.strftime("%Y-%m-%d"),
        }
        self.issued.setdefault(isbn, []).append(loan)
        print(f"'{book.title}' issued to {member_name}. Due back by {loan['due_date']}.")
        self.save_records()

    def return_book(self, isbn, member_name):
        book = self.books.get(isbn)
        if not book:
            print("No book found with that ISBN.")
            return

        loans = self.issued.get(isbn, [])
        matching = next((l for l in loans if l["member"].lower() == member_name.lower()), None)
        if not matching:
            print(f"No record of '{member_name}' having this book issued.")
            return

        loans.remove(matching)
        if not loans:
            self.issued.pop(isbn, None)
        book.copies_available = min(book.copies_available + 1, book.copies)
        print(f"'{book.title}' returned by {member_name}. Thank you!")
        self.save_records()

    def display_available_books(self):
        available = [b for b in self.books.values() if b.copies_available > 0]
        if not available:
            print("No books are currently available.")
            return
        print("\n--- Available Books ---")
        for b in sorted(available, key=lambda x: x.title.lower()):
            print(b)
        print("-----------------------\n")

    def display_all_books(self):
        if not self.books:
            print("The library has no books yet.")
            return
        print("\n--- All Books in Library ---")
        for b in sorted(self.books.values(), key=lambda x: x.title.lower()):
            print(b)
        print("-----------------------------\n")


# ---------------------------- CLI ----------------------------

def print_menu():
    print("""
========== Library Management System ==========
1. Add a book
2. Search for a book
3. Issue a book
4. Return a book
5. Display available books
6. Display all books
7. Save records
8. Exit
=================================================""")


def prompt_int(prompt, default=1):
    val = input(prompt).strip()
    if not val:
        return default
    try:
        return int(val)
    except ValueError:
        print("Invalid number, using default value.")
        return default


def main():
    library = Library()

    while True:
        print_menu()
        choice = input("Enter your choice (1-8): ").strip()

        if choice == "1":
            print("\n-- Add a Book --")
            isbn = input("ISBN: ").strip()
            title = input("Title: ").strip()
            author = input("Author: ").strip()
            copies = prompt_int("Number of copies [1]: ", default=1)
            if isbn and title and author:
                library.add_book(isbn, title, author, copies)
            else:
                print("ISBN, title, and author are all required.")

        elif choice == "2":
            print("\n-- Search for a Book --")
            keyword = input("Enter title, author, or ISBN keyword: ").strip()
            results = library.search_books(keyword)
            if results:
                print(f"\nFound {len(results)} result(s):")
                for b in results:
                    print(b)
            else:
                print("No matching books found.")

        elif choice == "3":
            print("\n-- Issue a Book --")
            isbn = input("ISBN of the book: ").strip()
            member = input("Member name: ").strip()
            if isbn and member:
                library.issue_book(isbn, member)
            else:
                print("ISBN and member name are required.")

        elif choice == "4":
            print("\n-- Return a Book --")
            isbn = input("ISBN of the book: ").strip()
            member = input("Member name: ").strip()
            if isbn and member:
                library.return_book(isbn, member)
            else:
                print("ISBN and member name are required.")

        elif choice == "5":
            library.display_available_books()

        elif choice == "6":
            library.display_all_books()

        elif choice == "7":
            library.save_records()

        elif choice == "8":
            library.save_records()
            print("Goodbye!")
            break

        else:
            print("Invalid choice. Please enter a number between 1 and 8.")


if __name__ == "__main__":
    main()


========== Library Management System ==========
1. Add a book
2. Search for a book
3. Issue a book
4. Return a book
5. Display available books
6. Display all books
7. Save records
8. Exit
Enter your choice (1-8): 2

-- Search for a Book --
Enter title, author, or ISBN keyword:  George C. Clark Jr. and J. Bibb Cain

Found 1 result(s):
ISBN: 978-0-306-40615-7 | 'Error-Correction Coding for Digital Communications.Book' by George C. Clark Jr. and J. Bibb Cain | Available: 2000/2000

========== Library Management System ==========
1. Add a book
2. Search for a book
3. Issue a book
4. Return a book
5. Display available books
6. Display all books
7. Save records
8. Exit
Enter your choice (1-8): 7
Records saved to '/content/library_data.json'.

========== Library Management System ==========
1. Add a book
2. Search for a book
3. Issue a book
4. Return a book
5. Display available books
6. Display all books
7. Save records
8. Exit
Enter your choice (1-8): 4

-- Return a Book --
ISBN of the book